# 40 ? Phase 5: Legal Statute Identification (LSI)

Multi-label statute prediction from your four Phase-1 backbones. This notebook reuses the existing
`DocClassifier` and adapters; `lm_lib/lsi/` handles data, loss, evaluation, calibration, KD, and submission.
Only the supplied competition data and your Phase-1 checkpoints are used.

**Brief ?11:** all four architectures; at least four substantially different experiments;
at least three adaptation techniques, including **LoRMA**, plus **one smaller-student KD run**.
The editable experiment table below covers full, LoRMA, partial, and KD. It is a workflow template,
not evidence of completed experiments or tuned settings. Choose your settings and write your own analysis.

Primary metric: **macro-F1**. Also report micro-F1, subset accuracy, per-statute F1/support, parameters,
and parameter counts. No softmax or argmax: every statute is an independent binary target.

The bundled brief lists Phase 5 as October 22?26, 2026. This notebook follows the requested Phase 5 scope.
See `CONTEXT.md` for the repository map and limitations. Runs go to `MyDrive/cs781/lsi_runs/`.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
# Run dependency installs before importing torch/model modules.
%pip -q install numpy pandas tokenizers ijson "huggingface_hub[hf_xet]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, glob, zipfile, tempfile, importlib, shutil, time, json
from pathlib import Path
MYDRIVE = Path('/content/drive/MyDrive')
DRIVE_ROOT = MYDRIVE / 'cs781'
RUNS_DIR = DRIVE_ROOT / 'runs'
OUT_DIR = DRIVE_ROOT / 'lsi_runs'
P5 = Path('/content/p5')
P5.mkdir(exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional explicit folder or archive; leave None to find an updated Phase-5 code copy.
CODE_SOURCE = None
levels = ['', '*/', '*/*/', '*/*/*/']
if CODE_SOURCE is None:
    folders = [Path(h).parents[2] for level in levels
               for h in glob.glob(str(MYDRIVE / f'{level}lm_lib/lsi/trainer.py'))]
    archives = [Path(h) for level in levels for h in glob.glob(str(MYDRIVE / f'{level}cs781_code*.zip'))]
    valid_archives = []
    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            if 'lm_lib/lsi/trainer.py' in z.namelist():
                valid_archives.append(archive)
    candidates = [(p, (p / 'lm_lib/lsi/trainer.py').stat().st_mtime) for p in folders]
    candidates += [(p, p.stat().st_mtime) for p in valid_archives]
    if not candidates:
        raise FileNotFoundError('Upload updated lm_lib/, models/, adapters/ folders, or rebuild and upload cs781_code.zip.')
    CODE_SOURCE = max(candidates, key=lambda item: item[1])[0]
CODE_SOURCE = Path(CODE_SOURCE)
if CODE_SOURCE.is_file():
    CODE_DIR = Path(tempfile.mkdtemp(prefix='cs781_phase5_', dir='/content'))
    with zipfile.ZipFile(CODE_SOURCE) as z:
        z.extractall(CODE_DIR)
else:
    CODE_DIR = CODE_SOURCE
assert (CODE_DIR / 'lm_lib/lsi/trainer.py').is_file(), 'Selected code copy lacks Phase 5'
sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
# Re-running with changed source after imports requires a runtime restart.
print('Code:', CODE_SOURCE, '\nRuns:', OUT_DIR)

In [ ]:
from huggingface_hub import hf_hub_download, login

def get_hf_token():
    if os.environ.get('HF_TOKEN'):
        return os.environ['HF_TOKEN'].strip()
    token_file = CODE_DIR / 'hf_token.txt'
    if token_file.is_file():
        return token_file.read_text().strip()
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('Set HF_TOKEN in Colab Secrets or the environment, or provide hf_token.txt beside the code.') from exc

HF_TOKEN = get_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)
# These paths and the record schema are specified in the bundled brief ?11.3.
cache_dir = DRIVE_ROOT / 'data' / 'phase5'
cache_dir.mkdir(parents=True, exist_ok=True)
for relative in ['Train/train.json', 'Train/dev.json', 'Test/test.json']:
    name = Path(relative).name
    cached = cache_dir / name
    if not cached.is_file():
        downloaded = hf_hub_download('Exploration-Lab/CS-781-Capstone', f'Phase_5_LSI/{relative}',
                                     repo_type='dataset', token=HF_TOKEN)
        shutil.copyfile(downloaded, cached)
    shutil.copyfile(cached, P5 / name)
    print(name, f'{(P5 / name).stat().st_size / 1e6:.1f} MB')

In [ ]:
PHASE1 = {
    'A': RUNS_DIR / 'A_qwen_dense_full',
    'B': RUNS_DIR / 'B_encdec',
    'C': RUNS_DIR / 'C_gdn_hybrid',
    'D': RUNS_DIR / 'D_mla_moe',
}
# Edit these paths if your saved runs have different names.
for arch, path in PHASE1.items():
    print(arch, path, '| checkpoints:', sorted(p.name for p in path.glob('*.pt')))

# Pick one run at a time. Run A_full before D_kd. Use new names for revised experiments.
EXPERIMENTS = {
    'A_full': dict(arch='A', method='full', method_kwargs={}, run_name='A_full__spread4x512_3'),
    'B_lorma': dict(arch='B', method='lorma', method_kwargs={'r': 8, 'lorma_alpha': 16},
                    run_name='B_lorma__spread4x512'),
    'C_partial': dict(arch='C', method='partial', method_kwargs={'k': 4},
                      run_name='C_partial__spread4x512'),
    'D_kd': dict(arch='D', method='full', method_kwargs={}, run_name='D_kd_from_A__spread4x512_3',
                 kd_teacher=str(OUT_DIR / 'A_full__spread4x512_3')),
}
EXPERIMENT = 'D_kd'
selected = EXPERIMENTS[EXPERIMENT].copy()
ARCH = selected.pop('arch')
if ARCH == 'C':
    %pip -q install flash-linear-attention
# Install before the next cell imports the backbone. Restart if pip asks you to.

Inspect the released data before training. The label vocabulary is the sorted union of released train/dev
statute IDs (metadata only); it is saved with every run and never inferred from test labels. If organizers
supply a fuller official vocabulary, enter it in `label_ids=` below. The brief does not specify a label count
or guarantee contiguous IDs. Labels absent from training are flagged rather than silently removed.

In [ ]:
import importlib
from pathlib import Path

p = Path(ld.__file__)
source = p.read_text()
source = source.replace(
    '            if len(labels) != len(set(labels)):\n'
    '                raise ValueError(f"{ident}: duplicate statute IDs")\n'
    '            d["labels"] = sorted(labels)',
    '            d["labels"] = sorted(set(labels))'
)
p.write_text(source)
importlib.reload(ld)

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from lm_lib.lsi import data as ld
train_docs = ld.load_split(P5 / 'train.json')
dev_docs = ld.load_split(P5 / 'dev.json')
test_docs = ld.load_split(P5 / 'test.json', labeled=False)
ld.check_disjoint(train=train_docs, dev=dev_docs, test=test_docs)
LABEL_IDS = ld.label_vocab(train_docs, dev_docs)
counts = {}
for split, docs in [('train', train_docs), ('dev', dev_docs)]:
    counts[split] = Counter(v for doc in docs for v in doc['labels'])
    print(split, len(docs), 'docs | mean labels/doc:', round(np.mean([len(d['labels']) for d in docs]), 2))
print('test:', len(test_docs), '| label count:', len(LABEL_IDS), '| IDs:', LABEL_IDS)
label_counts = pd.DataFrame(counts).reindex(LABEL_IDS).fillna(0).astype(int)
label_counts.index.name = 'statute_id'
display(label_counts)
print('Dev-only labels:', sorted(set(counts['dev']) - set(counts['train'])))
for split, docs in [('train', train_docs), ('dev', dev_docs), ('test', test_docs)]:
    print(split, 'word-length percentiles [50, 90, 99, max]:',
          np.percentile([len(d['text'].split()) for d in docs], [50, 90, 99, 100]).astype(int))
# Keep the displayed counts, release raw text from RAM before the trainer reloads it.
del train_docs, dev_docs, test_docs

The values below inherit starting settings from Phase 4; **you choose and tune them**. `spread` samples
windows across a document, `last` keeps its end, and `first_last` selects its beginning/end. These are bounded
representations, not full-document coverage. `last`/`first_last` reuse the approximate character preselection
in `cjpe/data.py`; `spread` tokenizes the whole text before sampling.

`pos_weight='balanced'` uses capped negative/positive counts from the actual training subset only.
Selection uses fixed-threshold dev macro-F1. Optional `global`/`per_label` thresholds are fitted afterwards
on the best checkpoint's dev probabilities and saved for inference; their dev scores are optimistic because
the same dev set supplies calibration and evaluation. Rare labels without sufficient dev positives/negatives
retain the global threshold. Empty label predictions are allowed; there is no forced top-1 fallback.

`n_train > 0` is a random smoke-test subset and can omit rare statutes. Use `n_train=0` for final experiments.
For KD, match the teacher's chunking/normalization and tokenizer exactly. Student total parameters must be
smaller than the teacher's; the trainer checks actual saved model sizes, not architecture names.

In [ ]:
from lm_lib.lsi.trainer import LSIConfig, train_lsi, predict_test
cfg = LSIConfig(
    phase1_run=str(PHASE1[ARCH]), phase1_ckpt='latest',
    data_dir=str(P5), out_dir=str(OUT_DIR), label_ids=LABEL_IDS,
    **selected,
    strategy='spread', chunk_len=512, max_chunks=4,
    token_pooling='mean', chunk_agg='mean', normalize_text=True,
    n_train=0, batch_size=8, grad_accum=4, grad_checkpoint=False,
    epochs=1, lr=5e-5, head_lr=1e-3, weight_decay=0.01, warmup_frac=0.1, dropout=0.1,
    pos_weight='none', pos_weight_cap=20.0,
    threshold=0.5, threshold_mode='none', threshold_min_support=5,
    eval_every=0, kd_alpha=0.5, kd_temperature=2.0, stop_at='', seed=1337,
)
# OOM: reduce physical batch/chunks/length or enable checkpointing; increase grad_accum if appropriate.
# B uses encoder features; chunk_len must not exceed its pretrained src_len (usually 768).
# Run names must be new: existing runs are protected; training does not resume optimizer state.
run_dir = OUT_DIR / cfg.run_name
for name in ['latest.pt', 'tokenizer.json']:
    assert (PHASE1[ARCH] / name).is_file(), f'Missing Phase-1 asset: {PHASE1[ARCH] / name}'
if cfg.kd_teacher:
    assert (Path(cfg.kd_teacher) / 'best.pt').is_file(), 'Train the teacher first'
print(cfg)

In [ ]:
# Preflight the actual selected Phase-1 checkpoint, labels, padding, feature gradients and loss.
# This is a small forward/backward check, not a performance experiment.
import gc, torch
from lm_lib.lsi.trainer import build_classifier
from lm_lib.cjpe.trainer import _batches
from lm_lib.lsi.metrics import multi_hot
from lm_lib.amp import autocast
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
clf, tok, report = build_classifier(cfg, LABEL_IDS, DEVICE)
print(DEVICE, report)
small_docs = ld.load_split(P5 / 'train.json')[:2]
small = ld.prepare(small_docs, tok, cfg)
batch, ids, tm, cm = next(_batches(small, 2, cfg.chunk_len, tok.token_to_id('<|pad|>'), False, None))
with autocast(DEVICE):
    logits = clf(ids.to(DEVICE), tm.to(DEVICE), cm.to(DEVICE)).float()
y = torch.tensor(multi_hot(batch, LABEL_IDS), device=DEVICE)
loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y)
assert logits.shape == y.shape and torch.isfinite(loss)
loss.backward()
assert clf.head.weight.grad is not None and torch.isfinite(clf.head.weight.grad).all()
print('Preflight OK:', tuple(logits.shape), 'BCE:', float(loss.detach()))
del clf, tok, logits, y, loss, ids, tm, cm, small, small_docs, batch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
results = train_lsi(cfg)
print('Saved:', run_dir)
print('Fixed-threshold macro-F1:', results['best_dev_macro_f1'])
print('Calibrated macro-F1:', results['calibrated_dev_scores']['macro_f1'])

In [ ]:
# Predict from saved best.pt + saved thresholds; never read test labels.
# These functions rebuild the run, so this also works after restarting and redoing setup.
zip_path, test_preds = predict_test(str(run_dir), data_dir=str(P5), bs=cfg.batch_size)
print('Submission on Drive:', zip_path)
print('Preview:', test_preds[:3])
with zipfile.ZipFile(zip_path) as z:
    assert z.namelist() == ['prediction.json']
# Optional download:
# from google.colab import files
# files.download(zip_path)

In [ ]:
rows = []
for path in OUT_DIR.glob('*/results.json'):
    r = json.loads(path.read_text())
    fixed, tuned = r['fixed_dev_scores'], r['calibrated_dev_scores']
    rows.append(dict(run=r['run'], phase1=r['phase1_run'], method=r['method'],
                     kd=bool(r['kd_teacher']), n_train=r['n_train'],
                     fixed_macro_f1=fixed['macro_f1'], micro_f1=fixed['micro_f1'],
                     subset_accuracy=fixed['subset_accuracy'], calibrated_macro_f1=tuned['macro_f1'],
                     threshold_mode=r['threshold_mode'], total_params=r['total_params'],
                     trainable_pct=r['trainable_pct'],
                     stopped_early=r['stopped_early']))
summary = pd.DataFrame(rows)
if not summary.empty:
    summary = summary.sort_values('fixed_macro_f1', ascending=False)
    summary.to_csv(OUT_DIR / 'comparison.csv', index=False)
    display(summary)
    print('Coverage: distinct Phase-1 runs =', summary.phase1.nunique(),
          '| adaptation methods =', sorted(summary.method.unique()),
          '| LoRMA present =', bool((summary.method == 'lorma').any()),
          '| KD present =', bool(summary.kd.any()))
else:
    print('No completed experiments yet.')
print('Check all four architectures and export one submission for each. Counts do not prove methodological quality.')

In [ ]:
# Evidence for your own rare-statute/error analysis, without generating conclusions.
per_label = pd.read_json(run_dir / 'per_label.json')
display(per_label.sort_values(['support', 'label']))
thresholds = json.loads((run_dir / 'thresholds.json').read_text())
display(pd.DataFrame({'label': thresholds['label_ids'], 'threshold': thresholds['thresholds']}))

Record for your report (complete yourself):

- Which four Phase-1 checkpoints and adaptation techniques did you run? Which submission belongs to each?
- Compare fixed-threshold macro-F1, micro-F1 and subset accuracy alongside trainable parameter counts.
- Compare bounded context selection and pooling under a controlled setting; record what fraction of long cases is seen.
- Compare unweighted/weighted BCE and fixed/global/per-label thresholds. Identify rare-statute misses and co-occurrence patterns.
- Compare KD with its teacher and an undistilled student under matching inputs; discuss actual size and inference costs.
- Record seeds, subsets, early stops, exact configurations, references, and assistance used.

Limitations: no real scores exist until you run this on your checkpoints. Phase-1 weights and official data are
not bundled here. Local macro-F1 averages over the persisted released vocabulary; the brief says the private
evaluator derives its label dimension from references, so confirm its treatment of absent labels against the
official scorer if provided. Existing feature extraction does not pass token padding masks into backbone attention;
this is particularly relevant to B's bidirectional encoder. Right-padded chunks are masked for pooling only.
Best weights are saved on dev improvement; no optimizer resume is implemented for Phase 5.